In [19]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import OneHotEncoder

In [20]:
DATA_PATH=Path('data/CLEAN_AB_NYC_2019.csv')
df=pd.read_csv(DATA_PATH)
df

,name,host_id,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365,days_since_last_review,has_reviews
0,Clean & quiet apt home by the park,2787,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,0.21,6,365,262.0,1
1,Skylit Midtown Castle,2845,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,0.38,2,355,48.0,1
2,THE VILLAGE OF HARLEM....NEW YORK !,4632,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,0.00,1,365,-1.0,0
3,Cozy Entire Floor of Brownstone,4869,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,4.64,1,194,3.0,1
4,Entire Apt: Spacious Studio/Loft by central park,7192,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,0.10,1,0,231.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48879,Charming one bedroom - newly renovated rowhouse,8232441,Brooklyn,Bedford-Stuyvesant,40.67853,-73.94995,Private room,70,2,0,0.00,2,9,-1.0,0
48880,Affordable room in Bushwick/East Williamsburg,6570630,Brooklyn,Bushwick,40.70184,-73.93317,Private room,40,4,0,0.00,2,36,-1.0,0
48881,Sunny Studio at Historical Neighborhood,23492952,Manhattan,Harlem,40.81475,-73.94867,Entire home/apt,115,10,0,0.00,1,27,-1.0,0
48882,43rd St. Time Square-cozy single bed,30985759,Manhattan,Hell's Kitchen,40.75751,-73.99112,Shared room,55,1,0,0.00,6,2,-1.0,0


### Haversine 

In [21]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371 
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

df['dist_to_manhattan_center'] = haversine_km(df['latitude'], df['longitude'], 40.7580, -73.9855)
df['dist_to_jfk_km']           = haversine_km(df['latitude'], df['longitude'], 40.6413, -73.7781)
df['dist_to_brooklyn_bridge']  = haversine_km(df['latitude'], df['longitude'], 40.7061, -73.9969)

In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48884 entries, 0 to 48883
Data columns (total 18 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   name                            48868 non-null  str    
 1   host_id                         48884 non-null  int64  
 2   neighbourhood_group             48884 non-null  str    
 3   neighbourhood                   48884 non-null  str    
 4   latitude                        48884 non-null  float64
 5   longitude                       48884 non-null  float64
 6   room_type                       48884 non-null  str    
 7   price                           48884 non-null  int64  
 8   minimum_nights                  48884 non-null  int64  
 9   number_of_reviews               48884 non-null  int64  
 10  reviews_per_month               48884 non-null  float64
 11  calculated_host_listings_count  48884 non-null  int64  
 12  availability_365                48884 non-n

In [23]:
df[['number_of_reviews','reviews_per_month']]

,number_of_reviews,reviews_per_month
0,9,0.21
1,45,0.38
2,0,0.00
3,270,4.64
4,9,0.10
...,...,...
48879,0,0.00
48880,0,0.00
48881,0,0.00
48882,0,0.00


In [24]:
one=OneHotEncoder()
one_array=one.fit_transform(df[['room_type','neighbourhood_group']]).toarray()
one_df=pd.DataFrame(one_array,columns=one.get_feature_names_out(),index=df.index)
df=pd.concat([df,one_df],axis=1)

In [25]:
df = df.drop(columns=['room_type', 'neighbourhood_group'])

In [26]:
df['reviews_per_month'] = df['reviews_per_month'].fillna(0)

In [27]:
df['popularity_index'] = df['reviews_per_month'] * np.log1p(df['number_of_reviews'])

In [28]:
df['popularity_index']

0         0.483543
1         1.454884
2         0.000000
3        25.993831
4         0.230259
           ...    
48879     0.000000
48880     0.000000
48881     0.000000
48882     0.000000
48883     0.000000
Name: popularity_index, Length: 48884, dtype: float64

In [29]:
host_calc = df.groupby('host_id').size()
df['host_total_listings'] = df['host_id'].map(host_calc)
print((df['host_total_listings'] == df['calculated_host_listings_count']).all())

False


In [30]:
df = df.drop('host_total_listings', axis=1)
HOST_COL = 'calculated_host_listings_count'
df['reviews_per_listing'] = df['number_of_reviews'] / df[HOST_COL]

In [31]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48884 entries, 0 to 48883
Data columns (total 26 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   name                               48868 non-null  str    
 1   host_id                            48884 non-null  int64  
 2   neighbourhood                      48884 non-null  str    
 3   latitude                           48884 non-null  float64
 4   longitude                          48884 non-null  float64
 5   price                              48884 non-null  int64  
 6   minimum_nights                     48884 non-null  int64  
 7   number_of_reviews                  48884 non-null  int64  
 8   reviews_per_month                  48884 non-null  float64
 9   calculated_host_listings_count     48884 non-null  int64  
 10  availability_365                   48884 non-null  int64  
 11  days_since_last_review             48884 non-null  float64
 12  h

In [32]:
df['name'] = df['name'].fillna('')
df['name_length'] = df['name'].str.len()
df['name_word_count'] = df['name'].str.split().str.len()
df['kw_luxury'] = df['name'].str.lower().str.contains('luxur|premium|exclusive').astype(int)
df['kw_cozy']   = df['name'].str.lower().str.contains('cozy|charming|cute').astype(int)
df['kw_view']   = df['name'].str.lower().str.contains('view|skyline|rooftop').astype(int)
df['kw_central']= df['name'].str.lower().str.contains('central|downtown|midtown').astype(int)

In [33]:
df = df.drop(['host_id','name'], axis=1)

In [34]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48884 entries, 0 to 48883
Data columns (total 30 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   neighbourhood                      48884 non-null  str    
 1   latitude                           48884 non-null  float64
 2   longitude                          48884 non-null  float64
 3   price                              48884 non-null  int64  
 4   minimum_nights                     48884 non-null  int64  
 5   number_of_reviews                  48884 non-null  int64  
 6   reviews_per_month                  48884 non-null  float64
 7   calculated_host_listings_count     48884 non-null  int64  
 8   availability_365                   48884 non-null  int64  
 9   days_since_last_review             48884 non-null  float64
 10  has_reviews                        48884 non-null  int64  
 11  dist_to_manhattan_center           48884 non-null  float64
 12  d

In [35]:
df.to_csv('data/Features_AB_NYC_2019.csv',index=False)